In [13]:
import json
import random
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# -------------------------------
# Intent Dataset
# -------------------------------
intents = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": [
                "hello",
                "hi",
                "hey",
                "good morning",
                "good evening"
            ],
            "responses": [
                "Hello! How can I help you today?",
                "Hi there! Hope you are having a nice day. How can I assist?"
            ]
        },
        {
            "tag": "admission",
            "patterns": [
                "how can i get admission",
                "admission process",
                "how to enroll",
                "admission requirements"
            ],
            "responses": [
                "You can apply through the admission portal.",
                "Please visit our website for admission details."
            ]
        },
        {
            "tag": "fees",
            "patterns": [
                "what is the fee structure",
                "course fees",
                "tuition fee",
                "how much does it cost"
            ],
            "responses": [
                "Fee details are available on the fees section of our website."
            ]
        },
        {
            "tag": "hours",
            "patterns": [
                "working hours",
                "office timing",
                "support hours",
                "when are you open"
            ],
            "responses": [
                "Our office is open Monday to Friday, 9 AM to 5 PM."
            ]
        },
        {
            "tag": "bot_wellbeing",
            "patterns": [
                "how are you",
                "how is it going",
                "how are you doing",
                "how is your day",
                "what's up",
                "whats up"
            ],
            "responses": [
                "I'm doing great, thank you! How are you doing today?",
                "All good here! Hope your day is going beautifully as well!"
            ]
        },
        {
            "tag": "user_wellbeing_good",
            "patterns": [
                "i am fine",
                "i am doing good",
                "doing great",
                "im good",
                "im doing well",
                "having a good day"
            ],
            "responses": [
                "That's wonderful to hear! I'm glad you are doing well.",
                "Awesome! Keep up the great positive energy!"
            ]
        },
        {
            "tag": "user_wellbeing_bad",
            "patterns": [
                "i am tired",
                "feeling sad",
                "bad day",
                "im not doing well",
                "im stressed"
            ],
            "responses": [
                "I'm sorry to hear that. Take a deep breath, and remember to take some time for yourself!",
                "I understand. Hang in there! Sending you virtual support and good vibes."
            ]
        },
        {
            "tag": "goodbye",
            "patterns": [
                "bye",
                "goodbye",
                "see you later"
            ],
            "responses": [
                "Goodbye!",
                "Have a great day!"
            ]
        }
    ]
}

# -------------------------------
# Prepare Training Data
# -------------------------------
texts = []
labels = []

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        texts.append(pattern)
        labels.append(intent["tag"])

# -------------------------------
# Train Model
# -------------------------------
vectorizer = TfidfVectorizer(
    lowercase=True
)

X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)

# Optional: Save Model
pickle.dump(model, open("chatbot_model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

# -------------------------------
# Text Preprocessing
# -------------------------------
def preprocess(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

# -------------------------------
# Chatbot Prediction
# -------------------------------
def get_response(user_message):
    processed_message = preprocess(user_message)

    # 1. Quick exact keyword/pattern fallback check
    for intent in intents["intents"]:
        for pattern in intent["patterns"]:
            if preprocess(pattern) in processed_message or processed_message in preprocess(pattern):
                return {
                    "intent": intent["tagCustom" if "tagCustom" in intent else "tag"],
                    "confidence": 1.0,
                    "response": random.choice(intent["responses"])
                }

    # 2. Machine Learning Model Match
    X_test = vectorizer.transform([processed_message])
    probabilities = model.predict_proba(X_test)[0]
    confidence = max(probabilities)
    predicted_intent = model.predict(X_test)[0]

    # Fallback threshold
    if confidence < 0.40:
        return {
            "intent": "fallback",
            "confidence": round(float(confidence), 2),
            "response": "Sorry, I didn't understand your question."
        }

    for intent in intents["intents"]:
        target_tag = intent["tag"]
        if target_tag == predicted_intent:
            return {
                "intent": predicted_intent,
                "confidence": round(float(confidence), 2),
                "response": random.choice(intent["responses"])
            }

print("Model retrained with conversational life responses!")

Model retrained with conversational life responses!


### Test Your Chatbot Directly Below

Run the cell below to chat with your AI Student Support assistant directly inside the notebook without needing any external server links.

In [16]:
#@title Interactive Chatbot Test

user_input = "i am feeling stressed" #@param {type:"string"}

if user_input.strip():
    response_data = get_response(user_input)
    print(f"User: {user_input}")
    print(f"Bot: {response_data['response']} (Intent: {response_data['intent']}, Confidence: {response_data['confidence']})")
else:
    print("Please type a message in the input box above.")

User: i am feeling stressed
Bot: Sorry, I didn't understand your question. (Intent: fallback, Confidence: 0.29)
